In [ ]:
# Remove broken stack
!pip uninstall -y numpy pandas torch torchvision torchaudio transformers datasets peft accelerate bitsandbytes evaluate sacrebleu huggingface_hub triton

# Install clean compatible stack (FIXED)
!pip install -q \
  numpy==1.26.4 \
  pandas==2.2.2 \
  torch==2.2.2 \
  torchvision==0.17.2 \
  torchaudio==2.2.2 \
  transformers==4.44.2 \
  datasets==2.21.0 \
  peft==0.12.0 \
  accelerate==0.33.0 \
  bitsandbytes==0.43.1 \
  triton==2.2.0 \
  evaluate==0.4.2 \
  sacrebleu==2.4.3 \
  huggingface_hub==0.24.6 \
  sentencepiece

In [ ]:
!pip uninstall bitsandbytes
!pip install bitsandbytes==0.46.1

In [ ]:
# pip uninstall -y triton bitsandbytes

In [ ]:
# pip install -q triton==2.2.0 bitsandbytes==0.43.1

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd


In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

In [ ]:

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/shared/mBART_Finetune_Hinglish/pacman/bidirect"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
DRIVE_DATA_PATH = "/content/drive/MyDrive/shared/mBART_Finetune_Hinglish/pacman/cleaned_data"

In [ ]:

from datasets import load_dataset

TRAIN_FILE = f"{DRIVE_DATA_PATH}/train/Final_6M_CM_SelectedColumns_train.jsonl"

dataset = load_dataset("json", data_files={"data": TRAIN_FILE})["data"]

print(dataset)

### Control Initial Dataset Size for Debugging

To prevent processing the entire dataset during debugging, we'll introduce a variable to limit the number of initial rows taken from the raw dataset *before* the bidirectional expansion. This significantly speeds up initial development and testing.

In [ ]:
# Set a limit for the initial dataset size for faster debugging.
# Set to `None` or a very large number to process the full dataset.
INITIAL_DATA_LIMIT = 100000 # Example: Process only 50,000 rows from the original dataset for expansion.

if INITIAL_DATA_LIMIT is not None and INITIAL_DATA_LIMIT < len(dataset):
    print(f"Limiting initial dataset to {INITIAL_DATA_LIMIT} rows for expansion.")
    dataset = dataset.select(range(INITIAL_DATA_LIMIT))
else:
    print("Processing full initial dataset for expansion.")

print(dataset)

In [ ]:
import random
from datasets import Dataset

def make_bidirectional(examples_batch, forward_weight=0.7):
    # Initialize lists to collect data for each column
    input_texts = []
    target_texts = []
    src_langs = []
    tgt_langs = []

    # Iterate through each example in the batch by index
    for i in range(len(examples_batch["cm_text_r"])): # Changed from mat_text
        cm_text_r = examples_batch["cm_text_r"][i] # Changed from mat_text
        emb_text = examples_batch["emb_text"][i]

        # Always include forward direction (Hinglish to English)
        input_texts.append(cm_text_r)
        target_texts.append(emb_text)
        src_langs.append("hi_IN")
        tgt_langs.append("en_XX")

        # Include reverse with probability (1 - forward_weight) (English to Hinglish)
        if random.random() < (1 - forward_weight):
            input_texts.append(emb_text)
            target_texts.append(cm_text_r) # Changed from mat_text
            src_langs.append("en_XX")
            tgt_langs.append("hi_IN")

    return {
        "input_text": input_texts,
        "target_text": target_texts,
        "src_lang": src_langs,
        "tgt_lang": tgt_langs
    }

print("Starting dataset expansion...")
# Optimize dataset expansion using .map() and .flatten()
# This replaces the manual loop and list construction with a more efficient datasets library operation.
expanded_dataset = dataset.map(
    make_bidirectional,
    fn_kwargs={"forward_weight": 0.7},
    batched=True, # Process examples in batches, and expect a dictionary of lists for columns
    remove_columns=dataset.column_names, # Remove the original columns as new ones are created
    load_from_cache_file=False # Ensures the operation runs and shows progress
)

# No need for flatten() if map is used correctly with batched=True
dataset = expanded_dataset

print("Dataset expansion complete. Final dataset structure after bidirectional expansion:")
print(dataset)


In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, MBart50TokenizerFast
import torch

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
# First split: train + temp
split_1 = dataset.train_test_split(test_size=0.1, seed=42)

# Second split: validation + test
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

dataset = {
    "train": split_1["train"],
    "validation": split_2["train"],
    "test": split_2["test"]
}

print(dataset)

In [ ]:
raw_eval_dataset = dataset["validation"]
print("Defined raw_eval_dataset for compute_metrics:", raw_eval_dataset)

In [ ]:
# #  CURRENTLY SMALL DATA (FAST DEBUG)
# #  Increase later for real training

# dataset["train"] = dataset["train"].select(range(10000))
# dataset["validation"] = dataset["validation"].select(range(500))
# dataset["test"] = dataset["test"].select(range(500))

In [ ]:
# dataset["train"] = dataset["train"]
# dataset["validation"] = dataset["validation"]
# dataset["test"] = dataset["test"]

In [ ]:
max_length = 128

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]
    src_langs = examples["src_lang"]
    tgt_langs = examples["tgt_lang"]

    model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for inp, tgt, src, tgt_l in zip(inputs, targets, src_langs, tgt_langs):
        tokenizer.src_lang = src
        tokenizer.tgt_lang = tgt_l

        tokenized_input = tokenizer(
            inp,
            max_length=max_length,
            truncation=True,
            padding=False
        )

        tokenized_target = tokenizer(
            text_target=tgt,
            max_length=max_length,
            truncation=True,
            padding=False
        )

        labels_ids = [
            (token if token != tokenizer.pad_token_id else -100)
            for token in tokenized_target["input_ids"]
        ]

        model_inputs["input_ids"].append(tokenized_input["input_ids"])
        model_inputs["attention_mask"].append(tokenized_input["attention_mask"])
        model_inputs["labels"].append(labels_ids)

    return model_inputs

### Tokenization Step

**Note:** This cell tokenizes the dataset. Run this only if you need to perform fresh tokenization or if the dataset has changed. If you have previously saved tokenized data to Drive, you can skip this cell and use the loading cell below to retrieve it directly, saving time.

In [ ]:
# raw_eval_dataset = dataset["validation"]

# tokenized_datasets = {
#     split: dataset[split].map(
#         preprocess_function,
#         batched=True,
#         remove_columns=dataset[split].column_names
#     )
#     for split in ["train", "validation", "test"]
# }

### Save Tokenized Data to Drive

Saving the tokenized datasets to Google Drive ensures that you can quickly load them in future sessions without re-running the tokenization steps. This is particularly useful for large datasets.

In [ ]:

# from datasets import load_from_disk
# import os

# TOKENIZED_DATA_DIR = os.path.join(OUTPUT_DIR, "tokenized_data")
# os.makedirs(TOKENIZED_DATA_DIR, exist_ok=True)

# for split, dataset_split in tokenized_datasets.items():
#     split_path = os.path.join(TOKENIZED_DATA_DIR, split)
#     dataset_split.save_to_disk(split_path)
#     print(f"Saved {split} dataset to {split_path}")

# print("All tokenized datasets saved to Drive.")

In a future session, you can load the tokenized datasets like this:

In [ ]:
from datasets import load_from_disk
import os

# Define OUTPUT_DIR if it's not already defined (e.g., in a fresh session)
# This assumes OUTPUT_DIR is a global variable from earlier steps.
# If running in a new session, you might need to define it explicitly first.
# For now, we'll assume it might not be defined from previous runs of specific cells.
if 'OUTPUT_DIR' not in locals() and 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = "/content/drive/MyDrive/mBART_Finetune_Hinglish/pacman/bidirect"

# Assuming OUTPUT_DIR is defined and mounted
TOKENIZED_DATA_DIR = os.path.join(OUTPUT_DIR, "tokenized_data")

# This code loads the tokenized datasets from disk (Google Drive)
loaded_tokenized_datasets = {
    "train": load_from_disk(os.path.join(TOKENIZED_DATA_DIR, "train")),
    "validation": load_from_disk(os.path.join(TOKENIZED_DATA_DIR, "validation")),
    "test": load_from_disk(os.path.join(TOKENIZED_DATA_DIR, "test"))
}

# Assign to the 'tokenized_datasets' variable for consistency with the rest of the notebook
tokenized_datasets = loaded_tokenized_datasets

print("Tokenized datasets loaded from disk.")
print(tokenized_datasets)

In [ ]:
test_dataset = dataset['test']
hi_en_pairs = [ex for ex in test_dataset if ex['src_lang'] == 'hi_IN' and ex['tgt_lang'] == 'en_XX']
print(f'Total pairs in test split: {len(test_dataset)}')
print(f'Hinglish to English pairs: {len(hi_en_pairs)}')
print(f'English to Hinglish pairs: {len(test_dataset) - len(hi_en_pairs)}')

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

if 'model' not in globals():
    raise RuntimeError("Model not loaded. Please re-run Cell 9 first.")


model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "out_proj", "fc1", "fc2", "lm_head"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Fix: reload torchvision to avoid circular import after reinstall
import importlib, sys
if 'torchvision' in sys.modules:
    importlib.reload(sys.modules['torchvision'])

import evaluate
import numpy as np

# Install missing dependency for bertscore
!pip install bert_score

# Load metrics
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Filter for Hinglish -> English pairs only based on current raw_eval_dataset
    filtered_preds = []
    filtered_labels = []
    filtered_sources = []

    for pred, label, example in zip(decoded_preds, decoded_labels, raw_eval_dataset):
        if example["src_lang"] == "hi_IN" and example["tgt_lang"] == "en_XX":
            filtered_preds.append(pred)
            filtered_labels.append(label)
            filtered_sources.append(example["input_text"])

    if len(filtered_preds) == 0:
        return {"bleu": 0.0, "chrf": 0.0, "bertscore": 0.0}

    # 1. SacreBLEU
    bleu_results = bleu.compute(predictions=filtered_preds, references=[[l] for l in filtered_labels])

    # 2. chrF
    chrf_results = chrf.compute(predictions=filtered_preds, references=[[l] for l in filtered_labels])

    # 3. BERTScore (using distilbert-base-uncased for speed, or roberta-large for accuracy)
    bert_results = bertscore.compute(predictions=filtered_preds, references=filtered_labels, lang="en")

    return {
        "bleu": bleu_results["score"],
        "chrf": chrf_results["score"],
        "bertscore_f1": np.mean(bert_results["f1"])
    }

In [ ]:
from transformers import Seq2SeqTrainingArguments
import os

training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "updated_checkpoint"), # Updated to include subfolder

    # Evaluation
    eval_strategy="no", # Changed to 'no' to skip validation evaluation
    # eval_steps=2001, # Removed as evaluation is disabled
    save_strategy="steps", # Keep saving based on steps
    save_steps=500,
    save_total_limit=15,

    # load_best_model_at_end=False, # Removed as evaluation is disabled
    # metric_for_best_model="bleu", # Removed as evaluation is disabled
    # greater_is_better=True, # Removed as evaluation is disabled

    # Learning
    learning_rate=5e-5,
    warmup_steps=350,
    lr_scheduler_type="cosine",

    # Batch
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,

    # Training
    weight_decay=0.01,
    num_train_epochs=2,

    # Generation
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=5,

    # Optimization
    fp16=True,
    optim="paged_adamw_8bit",

    # Logging
    logging_steps=500,

    push_to_hub=False
)

In [ ]:
# training_args.eval_steps

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
# # Re-initialize trainer with new args if needed, then train
# trainer.args = training_args
# trainer.train()

# # Removed explicit validation evaluation as it's handled by test set evaluation if desired.

In [ ]:
import os

def find_latest_checkpoint(output_dir):
    if not os.path.exists(output_dir):
        print(f"Output directory does not exist: {output_dir}")
        return None

    checkpoints = []
    for item in os.listdir(output_dir):
        full_path = os.path.join(output_dir, item)
        if os.path.isdir(full_path) and item.startswith("checkpoint-"):
            checkpoints.append(full_path)

    if not checkpoints:
        print("No checkpoints found in the output directory.")
        return None

    # Sort checkpoints by step number to find the latest one
    checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
    latest_checkpoint = checkpoints[-1]
    print(f"Found latest checkpoint: {latest_checkpoint}")
    return latest_checkpoint

def check_checkpoints_for_trainer_state(base_output_dir):
    updated_checkpoint_dir = os.path.join(base_output_dir, "updated_checkpoint")
    print(f"Checking checkpoint directories in: {updated_checkpoint_dir}")

    if not os.path.exists(updated_checkpoint_dir):
        print(f"Directory not found: {updated_checkpoint_dir}")
        return

    found_any = False
    for item in os.listdir(updated_checkpoint_dir):
        full_path = os.path.join(updated_checkpoint_dir, item)
        if os.path.isdir(full_path) and item.startswith("checkpoint-"):
            found_any = True
            trainer_state_path = os.path.join(full_path, "trainer_state.json")
            if os.path.exists(trainer_state_path):
                print(f"  - Checkpoint '{item}': trainer_state.json EXISTS")
            else:
                print(f"  - Checkpoint '{item}': trainer_state.json DOES NOT EXIST")

    if not found_any:
        print("No checkpoint directories found within 'updated_checkpoint'.")

latest_checkpoint_path = find_latest_checkpoint(os.path.join(OUTPUT_DIR, "updated_checkpoint"))

# Call the new function to check for trainer_state.json in all checkpoints
check_checkpoints_for_trainer_state(OUTPUT_DIR)


In [ ]:
import os

# # Re-initialize trainer with new args if needed, then train
# trainer.args = training_args

# Check for the manual resumable checkpoint first, as it was explicitly saved with trainer_state.json
MANUAL_CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "manual_resumable_checkpoint_custom")
valid_manual_checkpoint = None
if os.path.exists(MANUAL_CHECKPOINT_DIR) and os.path.exists(os.path.join(MANUAL_CHECKPOINT_DIR, "trainer_state.json")):
    valid_manual_checkpoint = MANUAL_CHECKPOINT_DIR

# Now, use a more robust way to select the checkpoint
checkpoint_to_resume_from = None

if valid_manual_checkpoint:
    checkpoint_to_resume_from = valid_manual_checkpoint
    print(f"Prioritizing and resuming training from manual checkpoint: {checkpoint_to_resume_from}")
elif latest_checkpoint_path and os.path.exists(os.path.join(latest_checkpoint_path, "trainer_state.json")):
    # Only use latest_checkpoint_path if trainer_state.json exists within it
    checkpoint_to_resume_from = latest_checkpoint_path
    print(f"Resuming training from automatically found checkpoint: {checkpoint_to_resume_from}")

if checkpoint_to_resume_from:
    trainer.train(resume_from_checkpoint=checkpoint_to_resume_from)
else:
    print("No valid checkpoint found with trainer_state.json, starting training from scratch.")
    trainer.train()

# Removed explicit validation evaluation as it's handled by test set evaluation if desired.